In [1]:
from functools import cached_property

import torch
import torch.nn as nn

In [2]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()

        # activations
        self.cnn_activation = nn.ReLU()
        self.fc_activation = nn.Tanh()

        # conv layers
        self.conv1 = nn.Conv2d(
            in_channels=1, 
            out_channels=8, 
            kernel_size=9, 
            stride=1, 
            padding=4
        )
        self.conv2 = nn.Conv2d(
            in_channels=self.conv1.out_channels, 
            out_channels=64, 
            kernel_size=5, 
            stride=2,
            padding=2
        )
        self.conv3 = nn.Conv2d(
            in_channels=self.conv2.out_channels, 
            out_channels=32, 
            kernel_size=5, 
            stride=2,
            padding=2
        )
        self.conv4 = nn.Conv2d(
            in_channels=self.conv3.out_channels, 
            out_channels=128, 
            kernel_size=3, 
            stride=1,
            padding=1
        )
        self.conv5 = nn.Conv2d(
            in_channels=self.conv4.out_channels, 
            out_channels=128, 
            kernel_size=3, 
            stride=1,
            padding=1
        )
        self.conv6 = nn.Conv2d(
            in_channels=self.conv5.out_channels, 
            out_channels=256, 
            kernel_size=3, 
            stride=1,
            padding=1
        )
        self.conv7 = nn.Conv2d(
            in_channels=self.conv6.out_channels, 
            out_channels=256, 
            kernel_size=3, 
            stride=1,
            padding=1
        )
        
        # pooling
        self.maxpool1 = nn.MaxPool2d(kernel_size=5, stride=2, padding=2)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)

        # fully connected layers
        self.fc1 = nn.Linear(256 * 48 * 2, 256 * 48)

    def forward(self, x):
        x = x.unsqueeze(1)

        # low-level features
        x = self.cnn_activation(self.conv1(x))
        x = self.cnn_activation(self.conv2(x))
        x = self.maxpool1(x)
        
        x = self.cnn_activation(self.conv3(x))
        x = self.maxpool2(x)

        # high-level features
        x = self.cnn_activation(self.conv4(x))
        x = self.cnn_activation(self.conv5(x))
        x = self.maxpool2(x)

        x = self.cnn_activation(self.conv6(x))
        x = self.cnn_activation(self.conv7(x))
        x = self.maxpool2(x)

        # reshape for FC layers
        x = x.view(-1, self.fc1.in_features)

        x = self.fc_activation(self.fc1(x))
        return x

    @cached_property
    def numel(self):
        return sum(p.numel() for p in self.parameters())

    @cached_property
    def size(self):
        return f'{sum(p.nelement() * p.element_size() for p in self.parameters()) / 1024 ** 3:.3f}Gi'

In [3]:
model = ConvNet()
model.size

'1.129Gi'

In [4]:
batch_size = 32
input_batch = torch.randn(batch_size, 3072, 128)

output_batch = model(input_batch)
print(output_batch.shape)

torch.Size([32, 12288])
